# Merge Macro Data

This version is intentionally simple for a one-time project.

What it does:
- pulls FRED series from API
- reads local Freddie PMMS Excel
- reads local FHFA state HPI CSV
- reads local AQI county files and collapses them to `state-year median(Median AQI)`
- merges everything into the default or prepay modeling panel

Notes:
- `PMMS` is weekly, so it is averaged to month
- `FHFA state HPI` is quarterly, so each quarter value is copied to the 3 months in that quarter
- `AQI` is county-year, but your panel does not have county, so it is aggregated to `state-year`
- if you only want one AQI attribute, `Median AQI` is a reasonable choice

In [ ]:
import os
from pathlib import Path

import pandas as pd
import requests

ROOT = Path('/Users/jinyecai/Desktop/ML_Mortgage')
PROC = ROOT / 'processed_mortgage'
MACRO = PROC / 'macro_raw'
AQI_DIR = PROC / 'AQI'
OUTDIR = PROC / 'macro_merged'
OUTDIR.mkdir(parents=True, exist_ok=True)

PANEL_PATHS = {
    'default': PROC / 'panel_default_modeling_2016_2025.parquet',
    'prepay': PROC / 'panel_prepay_modeling_2016_2025.parquet',
}

PMMS_PATH = MACRO / 'freddie_pmms' / 'pmms_rates.xlsx'
FHFA_PATH = MACRO / 'fhfa' / 'fha_hpi_master.csv'

FRED_API_KEY = os.getenv('FRED_API_KEY', '')
FRED_SERIES = {
    'UNRATE': 'unrate',
    'CPIAUCSL': 'cpi',
    'FEDFUNDS': 'fedfunds',
    'GS10': 'gs10',
}

STATE_ABBR = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR', 'CALIFORNIA': 'CA',
    'COLORADO': 'CO', 'CONNECTICUT': 'CT', 'DELAWARE': 'DE', 'DISTRICT OF COLUMBIA': 'DC',
    'FLORIDA': 'FL', 'GEORGIA': 'GA', 'HAWAII': 'HI', 'IDAHO': 'ID', 'ILLINOIS': 'IL',
    'INDIANA': 'IN', 'IOWA': 'IA', 'KANSAS': 'KS', 'KENTUCKY': 'KY', 'LOUISIANA': 'LA',
    'MAINE': 'ME', 'MARYLAND': 'MD', 'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN',
    'MISSISSIPPI': 'MS', 'MISSOURI': 'MO', 'MONTANA': 'MT', 'NEBRASKA': 'NE', 'NEVADA': 'NV',
    'NEW HAMPSHIRE': 'NH', 'NEW JERSEY': 'NJ', 'NEW MEXICO': 'NM', 'NEW YORK': 'NY',
    'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH', 'OKLAHOMA': 'OK', 'OREGON': 'OR',
    'PENNSYLVANIA': 'PA', 'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC', 'SOUTH DAKOTA': 'SD',
    'TENNESSEE': 'TN', 'TEXAS': 'TX', 'UTAH': 'UT', 'VERMONT': 'VT', 'VIRGINIA': 'VA',
    'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV', 'WISCONSIN': 'WI', 'WYOMING': 'WY'
}


def to_yyyymm(date_series):
    dt = pd.to_datetime(date_series)
    return dt.dt.year * 100 + dt.dt.month


def load_panel(kind='default'):
    return pd.read_parquet(PANEL_PATHS[kind]).copy()


def get_fred_series(series_id, api_key=FRED_API_KEY):
    url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id': series_id,
        'api_key': api_key,
        'file_type': 'json'
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json()['observations'])[['date', 'value']].copy()
    df['value'] = pd.to_numeric(df['value'].replace('.', pd.NA), errors='coerce')
    df['rp_yyyymm'] = to_yyyymm(df['date'])
    return df.groupby('rp_yyyymm', as_index=False)['value'].last()


def load_fred_bundle(api_key=FRED_API_KEY):
    out = None
    for series_id, name in FRED_SERIES.items():
        df = get_fred_series(series_id, api_key=api_key).rename(columns={'value': name})
        out = df if out is None else out.merge(df, on='rp_yyyymm', how='outer')
    out = out.sort_values('rp_yyyymm').reset_index(drop=True)
    out['cpi_yoy'] = out['cpi'].pct_change(12)
    return out


def load_pmms():
    df = pd.read_excel(PMMS_PATH, header=5)
    df = df.rename(columns={'Unnamed: 0': 'week', '30 yr': 'pmms30'})
    df = df[['week', 'pmms30']].copy()
    df['week'] = pd.to_datetime(df['week'], errors='coerce', format='mixed')
    df = df.dropna(subset=['week']).copy()
    df['pmms30'] = pd.to_numeric(df['pmms30'], errors='coerce')
    df['rp_yyyymm'] = to_yyyymm(df['week'])
    return df.groupby('rp_yyyymm', as_index=False)['pmms30'].mean()


def load_fhfa_state_hpi():
    df = pd.read_csv(FHFA_PATH, low_memory=False)
    df = df[(df['level'] == 'State') & (df['frequency'] == 'quarterly') & (df['hpi_flavor'] == 'purchase-only')].copy()
    df['property_state'] = df['place_name'].str.upper().map(STATE_ABBR)
    quarter_months = {1: [1, 2, 3], 2: [4, 5, 6], 3: [7, 8, 9], 4: [10, 11, 12]}
    rows = []
    for _, r in df[['property_state', 'yr', 'period', 'index_nsa']].dropna().iterrows():
        for m in quarter_months[int(r['period'])]:
            rows.append({
                'property_state': r['property_state'],
                'rp_yyyymm': int(r['yr']) * 100 + m,
                'fhfa_hpi': r['index_nsa']
            })
    out = pd.DataFrame(rows)
    out = out.sort_values(['property_state', 'rp_yyyymm']).reset_index(drop=True)
    out['fhfa_hpi_yoy'] = out.groupby('property_state')['fhfa_hpi'].pct_change(12)
    return out


def load_aqi_state_year():
    files = sorted(AQI_DIR.glob('annual_aqi_by_county_*.csv'))
    parts = [pd.read_csv(f, usecols=['State', 'Year', 'Median AQI']) for f in files]
    df = pd.concat(parts, ignore_index=True)
    df['property_state'] = df['State'].str.upper().map(STATE_ABBR)
    out = (
        df.groupby(['property_state', 'Year'], as_index=False)['Median AQI']
          .median()
          .rename(columns={'Year': 'year', 'Median AQI': 'median_aqi'})
    )
    return out


def merge_macro(kind='default', api_key=FRED_API_KEY, save=True):
    panel = load_panel(kind)
    panel['year'] = panel['rp_yyyymm'] // 100

    fred = load_fred_bundle(api_key)
    pmms = load_pmms()
    hpi = load_fhfa_state_hpi()
    aqi = load_aqi_state_year()

    out = panel.merge(fred, on='rp_yyyymm', how='left')
    out = out.merge(pmms, on='rp_yyyymm', how='left')
    out = out.merge(hpi, on=['property_state', 'rp_yyyymm'], how='left')
    out = out.merge(aqi, on=['property_state', 'year'], how='left')
    out['mortgage_treasury_spread'] = out['pmms30'] - out['gs10']

    save_path = None
    if save:
        save_path = OUTDIR / f'panel_{kind}_with_macro.parquet'
        out.to_parquet(save_path, index=False)

    return out, save_path


FRED_API_KEY = "501431dcdd4d89dc0547ab37ac6ac7e2"
default_macro, default_path = merge_macro('default', api_key=FRED_API_KEY)
prepay_macro, prepay_path = merge_macro('prepay', api_key=FRED_API_KEY)

# Quick checks
default_macro[['rp_yyyymm', 'property_state', 'unrate', 'pmms30', 'fhfa_hpi', 'median_aqi']].head()
default_macro[['unrate', 'cpi', 'fedfunds', 'gs10', 'pmms30', 'fhfa_hpi', 'median_aqi']].isna().mean().sort_values()


/var/folders/zk/lmw40k6550g71ccqbjlkfnjw0000gn/T/ipykernel_3357/3535661168.py:75: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out['cpi_yoy'] = out['cpi'].pct_change(12)
